# Classification Review & Correction

Displays a classification CSV alongside the original benchmark entry and ground-truth answer for each row, so you can spot-check **and fix** the model's labels by hand.

**Model output vs. corrections:** the generated CSV (`CLASSIFICATION_FILE`) is treated as **read-only**. Your manual fixes are stored in a separate **overlay** file next to it — `<stem>.corrections.csv` (e.g. `base_classifications.corrections.csv`) — which holds only the rows that deviate from the model. On load, the overlay is merged on top of the model output to form the *effective* view. Re-running `classify_benchmark.py` regenerates the model CSV without touching your overlay.

**Controls (Cell 2):**
- Set `CATEGORY` to the benchmark folder name.
- Set `CLASSIFICATION_FILE` to the model CSV you want to review:
  - `base_classifications.csv` → `id, translatable_params, localizable_query, localizable_parameters`
  - `base_classifications_params.csv` → `id, translatable_params` (no localizable columns)
- Use `FILTER_*` options to narrow down what you see (e.g. `FILTER_translatable_params = "false"` to hunt for the under-classified ones). Filters apply to the *corrected* view.
- Run Cell 3 to load + merge the overlay, then Cell 4 to browse and correct.

**Editing (Cell 4):** each card has a checkbox per classification column (checked = `"true"`). Toggling a box **immediately writes to the overlay** (`✓ saved · ✎ overrides model`). Setting a card back to the model's original values removes it from the overlay (`matches model`). The `loc. query` / `loc. params` boxes are disabled while `translatable` is unchecked and their cells kept blank, matching the schema.

> Requires `ipywidgets` (already installed in the venv). Needs a widget-capable frontend (JupyterLab / VS Code / Notebook 7). Keep the overlay under version control to track/revert manual fixes.

In [9]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import json
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

NOTEBOOK_DIR = Path().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT    = PACKAGE_ROOT / "data" / "benchmarks"

print(f"Package root: {PACKAGE_ROOT}")

Package root: C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl


In [10]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
CATEGORY = "multiple"   # benchmark folder under data/benchmarks/

# Which classification CSV to review (under data/benchmarks/<CATEGORY>/):
#   "base_classifications.csv"        -> id, translatable_params, localizable_query, localizable_parameters
#   "base_classifications_params.csv" -> id, translatable_params
CLASSIFICATION_FILE = "base_classifications_params.csv"

# ── Filters (set to None to disable) ─────────────────────────────────────────
# Show only entries where a classification column equals a specific value.
# Filters for columns not present in the chosen file are simply ignored.
FILTER_translatable_params    = None   # "true" | "false" | None
FILTER_localizable_query      = None   # "true" | "false" | None  (full file only)
FILTER_localizable_parameters = None   # "true" | "false" | None  (full file only)

# Show only entries whose ID contains this string (e.g. "multiple_42"), or None:
FILTER_id = None

# Maximum number of entries to display (None = all):
MAX_DISPLAY = None

In [11]:
# ── Cell 3: Load data (model output + corrections overlay) ────────────────────
bench_dir    = DATA_ROOT / CATEGORY
source_path  = bench_dir / "eng_base.json"
answer_path  = bench_dir / "possible_answer" / "eng_base.json"
csv_path     = bench_dir / CLASSIFICATION_FILE                       # model output (read-only)
overlay_path = csv_path.with_name(csv_path.stem + ".corrections.csv")  # human edits live here

for p in (source_path, answer_path, csv_path):
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

entries = {e["id"]: e for e in load_jsonl(source_path)}
answers = {a["id"]: a for a in load_jsonl(answer_path)}

# Model output — never modified by this notebook. Strings, blanks kept as "".
base_df = pd.read_csv(csv_path, dtype=str).fillna("")

if "translatable_params" not in base_df.columns:
    raise ValueError(
        f"{csv_path.name} has no 'translatable_params' column (found {list(base_df.columns)}). "
        "This notebook expects the new schema (translatable_params = 'true'/'false')."
    )

# Which classification columns this file actually has (params file lacks localizable_*).
CLASS_COLUMNS = [
    c for c in ("translatable_params", "localizable_query", "localizable_parameters")
    if c in base_df.columns
]

# Per-id model values, used to tell whether a correction actually deviates.
base_vals = {r["id"]: {c: r[c] for c in CLASS_COLUMNS} for r in base_df.to_dict("records")}

# Load existing corrections overlay (id -> {col: value}); only deviating rows live here.
corrections = {}
if overlay_path.exists():
    for r in pd.read_csv(overlay_path, dtype=str).fillna("").to_dict("records"):
        corrections[r["id"]] = {c: r.get(c, "") for c in CLASS_COLUMNS}

# Effective view = model output with corrections applied on top.
clf_df = base_df.copy()
for rid, vals in corrections.items():
    for c, v in vals.items():
        clf_df.loc[clf_df["id"] == rid, c] = v

print(f"Model file  : {csv_path.name}")
print(f"Overlay     : {overlay_path.name}  ({len(corrections)} correction(s) loaded)")
print(f"Entries     : {len(entries)}")
print(f"Answers     : {len(answers)}")
print(f"Classified  : {len(clf_df)}")

# Effective value distribution per classification column (after corrections)
for col in CLASS_COLUMNS:
    counts = clf_df[col].value_counts(dropna=False).to_dict()
    print(f"  {col}: {counts}")

Model file  : base_classifications_params.csv
Overlay     : base_classifications_params.corrections.csv  (0 correction(s) loaded)
Entries     : 200
Answers     : 200
Classified  : 200
  translatable_params: {'true': 122, 'false': 78}


In [12]:
# ── Cell 4: Browse & correct ──────────────────────────────────────────────────
# Each card shows the entry plus a checkbox per classification column. Ticking a
# box (checked = "true") records a correction and immediately writes it to the
# overlay file (csv_path.stem + ".corrections.csv") — the model output CSV is left
# untouched. Setting a card back to the model's values removes it from the overlay.
# For the full file, the localizable_* boxes are disabled while translatable_params
# is unticked and their cells are kept blank, matching the CSV schema.

CARD_BG     = "#1e1e1e"
CARD_BORDER = "#3a3a3a"
SECTION_BG  = "#2a2a2a"
TITLE_COLOR = "#e0e0e0"
LABEL_COLOR = "#aaaaaa"
TEXT_COLOR  = "#d4d4d4"
USER_COLOR  = "#7eb8f7"
ASST_COLOR  = "#aaaaaa"

TRANS_COL   = "translatable_params"
LOC_COLS    = [c for c in ("localizable_query", "localizable_parameters") if c in CLASS_COLUMNS]
SHORT_LABEL = {
    "translatable_params":    "translatable",
    "localizable_query":      "loc. query",
    "localizable_parameters": "loc. params",
}

def resolve_id(csv_id: str):
    """Map a CSV id to the matching entry/answer key.

    langasync submits batches with sequential integer indices (0, 1, 2, …)
    instead of our entry IDs.  We try three strategies:
      1. Direct match (csv_id == entry key)          e.g. "multiple_0"
      2. Category-prefixed  (CATEGORY + "_" + csv_id) e.g. "multiple_0"
      3. Numeric index into the sorted entry list
    """
    if csv_id in entries:
        return csv_id
    prefixed = f"{CATEGORY}_{csv_id}"
    if prefixed in entries:
        return prefixed
    try:
        idx = int(csv_id)
        return sorted(entries.keys())[idx]
    except (ValueError, IndexError):
        return None

def format_ground_truth(ground_truth):
    lines = []
    for call in ground_truth:
        for func_name, params in call.items():
            lines.append(f'<b style="color:{TITLE_COLOR}">{func_name}</b>')
            for param, values in params.items():
                display_val = next(
                    (v for v in values if v != "" and v is not None),
                    values[0] if values else ""
                )
                lines.append(
                    f'&nbsp;&nbsp;<code style="color:#ce9178">{param}</code>'
                    f' = <code style="color:#b5cea8">{json.dumps(display_val, ensure_ascii=False)}</code>'
                )
    return "<br>".join(lines)

def format_query(entry):
    parts = []
    for turn in entry.get("question", []):
        for msg in turn:
            role  = msg.get("role", "?")
            color = USER_COLOR if role == "user" else ASST_COLOR
            parts.append(
                f'<span style="color:{color};font-weight:600">{role}:</span> '
                f'<span style="color:{TEXT_COLOR}">{msg["content"]}</span>'
            )
    return "<br>".join(parts)

def format_functions(entry):
    lines = []
    for func in entry.get("function", []):
        name   = func.get("name", "?")
        params = func.get("parameters", {}).get("properties", {})
        req    = set(func.get("parameters", {}).get("required", []))
        param_strs = []
        for p, pdef in params.items():
            star = "*" if p in req else ""
            extra = ""
            if pdef.get("enum"):
                extra = ' <span style="color:#888">[' + " | ".join(map(str, pdef["enum"])) + "]</span>"
            param_strs.append(
                f'<code style="color:#ce9178">{p}{star}</code>'
                f'<span style="color:#888">: {pdef.get("type","any")}</span>{extra}'
            )
        lines.append(f'<b style="color:#dcdcaa">{name}</b>({", ".join(param_strs)})')
    return "<br>".join(lines)

def render_detail(label, entry, answer):
    gt_html    = format_ground_truth(answer["ground_truth"]) if answer else "<i style='color:#888'>no answer</i>"
    query_html = format_query(entry)     if entry else "<i style='color:#888'>no entry</i>"
    func_html  = format_functions(entry) if entry else "<i style='color:#888'>—</i>"
    return f"""
<div style="border:1px solid {CARD_BORDER};border-radius:8px 8px 0 0;border-bottom:none;
            padding:16px 16px 10px;font-family:sans-serif;background:{CARD_BG}">
  <div style="font-size:1.05em;font-weight:700;color:{TITLE_COLOR};margin-bottom:10px">{label}</div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;font-size:0.9em">
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">QUERY</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.6">{query_html}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">FUNCTIONS</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{func_html}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">GROUND TRUTH</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{gt_html}</div>
    </div>
  </div>
</div>"""

def _status(saved, overridden):
    prefix = '<span style="color:#5fd98a">✓ saved</span> · ' if saved else ""
    tag = ('<span style="color:#e0b341">✎ overrides model</span>' if overridden
           else '<span style="color:#666">matches model</span>')
    return f'<span style="font-size:0.8em">{prefix}{tag}</span>'

def save_overlay():
    """Write the corrections overlay (only rows that deviate from the model)."""
    rows = [{"id": rid, **vals} for rid, vals in corrections.items()]
    out = pd.DataFrame(rows, columns=["id"] + CLASS_COLUMNS)
    if len(out):
        out = out.sort_values("id", key=lambda s: s.str.split("_").str[-1].astype(int))
    out.to_csv(overlay_path, index=False)

def make_card(row):
    rid     = row["id"]
    real_id = resolve_id(rid)
    entry   = entries.get(real_id) if real_id else None
    answer  = answers.get(real_id) if real_id else None
    label   = real_id or rid

    detail = widgets.HTML(render_detail(label, entry, answer))
    status = widgets.HTML(_status(False, rid in corrections))

    boxes = {
        col: widgets.Checkbox(
            value=(row.get(col, "") == "true"),
            description=SHORT_LABEL.get(col, col),
            indent=False,
            layout=widgets.Layout(width="auto", margin="0 16px 0 0"),
        )
        for col in CLASS_COLUMNS
    }
    # localizable_* only meaningful while translatable_params is true
    for col in LOC_COLS:
        boxes[col].disabled = not boxes[TRANS_COL].value

    def commit(_change=None):
        trans_on = boxes[TRANS_COL].value
        new = {TRANS_COL: "true" if trans_on else "false"}
        for col in LOC_COLS:
            boxes[col].disabled = not trans_on
            new[col] = ("true" if boxes[col].value else "false") if trans_on else ""

        # Only keep an overlay entry when it actually deviates from the model.
        if all(new[c] == base_vals[rid][c] for c in CLASS_COLUMNS):
            corrections.pop(rid, None)
            overridden = False
        else:
            corrections[rid] = new
            overridden = True

        for c, v in new.items():                       # keep the effective view in sync
            clf_df.loc[clf_df["id"] == rid, c] = v
        save_overlay()
        status.value = _status(True, overridden)

    for cb in boxes.values():
        cb.observe(commit, names="value")

    controls = widgets.HBox(
        [widgets.HTML(f'<span style="color:{LABEL_COLOR};font-size:0.85em;margin-right:6px">set:</span>')]
        + list(boxes.values()) + [status],
        layout=widgets.Layout(
            padding="8px 16px", flex_flow="row wrap", align_items="center",
            border=f"1px solid {CARD_BORDER}", border_radius="0 0 8px 8px",
        ),
    )
    return widgets.VBox([detail, controls], layout=widgets.Layout(margin="0 0 14px 0"))

# ── Apply filters (pandas, on the effective/corrected view) ───────────────────
filtered = clf_df
for col, val in [
    ("translatable_params",    FILTER_translatable_params),
    ("localizable_query",      FILTER_localizable_query),
    ("localizable_parameters", FILTER_localizable_parameters),
]:
    if val is not None and col in filtered.columns:
        filtered = filtered[filtered[col] == val]
if FILTER_id is not None:
    filtered = filtered[filtered["id"].str.contains(FILTER_id, na=False)]

shown = filtered.head(MAX_DISPLAY) if MAX_DISPLAY else filtered
print(f"Showing {len(shown)} of {len(filtered)} matching entries "
      f"(total classified: {len(clf_df)})  —  corrections save to {overlay_path.name}")

if len(shown):
    display(widgets.VBox([make_card(row) for row in shown.to_dict("records")]))
else:
    print("No entries match the current filters.")

Showing 200 of 200 matching entries (total classified: 200)  —  corrections save to base_classifications_params.corrections.csv
